# 01 — RAG Bagian 1: Chunking dan Retrieval

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/09-rag/01_chunking_dan_retrieval.ipynb)

LLM hanya tahu isi data pelatihannya. LLM tidak tahu kebijakan internal perusahaanmu, dokumen terbaru, atau data pelanggan. Melatih ulang (fine-tuning) mahal dan tidak praktis untuk informasi yang sering berubah. **Retrieval-Augmented Generation (RAG)** menyelesaikannya dengan cara sederhana: **cari** potongan dokumen yang relevan, lalu **berikan** ke LLM sebagai konteks saat menjawab.

```
INGEST (sekali)   : dokumen → potong (chunking) → embedding → index
SAAT BERTANYA     : pertanyaan → cari chunk relevan (retrieval) → [rerank] → prompt LLM + chunk → jawaban + sumber
```

Kualitas jawaban RAG sangat bergantung pada retrieval: jika chunk yang tepat tidak ditemukan, LLM terbaik pun tidak bisa menjawab dengan benar. Karena itu notebook ini fokus pada separuh pertama pipeline, **tanpa LLM**, dan setiap pilihan **diukur**:
1. Tiga strategi **chunking**.
2. **Dense retrieval** dengan embedding (lanjutan modul 07).
3. **Metrik retrieval**: recall@k dan MRR.
4. **BM25** (pencarian kata kunci) dan **hybrid search** (RRF).
5. **Reranking** dengan cross-encoder.

**Korpus:** 10 dokumen panduan layanan **Toko Elektronik Nusantara (TEN)**, toko **fiktif** yang ditulis khusus untuk modul ini (kebijakan pengembalian, garansi, pengiriman, pembayaran, program member, dan lain-lain), ditambah 27 pertanyaan uji beserta jawaban acuan dan kutipan **bukti** dari dokumen. Karena tokonya fiktif, LLM tidak mungkin tahu jawabannya dari data pretraining, sehingga manfaat RAG terlihat jelas. File ada di folder [`korpus/`](korpus/) dan diunduh otomatis jika notebook dijalankan di Colab.

**Model:** embedding [`intfloat/multilingual-e5-small`](https://huggingface.co/intfloat/multilingual-e5-small) (MIT) dan reranker [`BAAI/bge-reranker-v2-m3`](https://huggingface.co/BAAI/bge-reranker-v2-m3) (Apache-2.0, unduhan ±2 GB). GPU mempercepat reranking, tetapi CPU tetap cukup.

## 0. Instalasi dan Data

In [ ]:
%pip install -q sentence-transformers rank_bm25 pandas matplotlib

In [ ]:
import json
import os
import re
import urllib.request
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer

QUICK_RUN = os.environ.get("QUICK_RUN") == "1"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device} | QUICK_RUN: {QUICK_RUN}")

REPO_RAW = "https://raw.githubusercontent.com/riki-profile/ai-engineer-journey/main/09-rag/korpus"
FILE_KORPUS = [f"dokumen/{n}.md" for n in [
    "01_pengembalian_barang", "02_garansi", "03_pengiriman",
    "04_pembayaran", "05_program_member", "06_tukar_tambah",
    "07_servis_berbayar", "08_akun_dan_keamanan", "09_promo_dan_voucher",
    "10_privasi_data"]] + ["pertanyaan_uji.jsonl"]


def muat_korpus(folder=Path("korpus")):
    """Baca korpus dari folder lokal (repo di-clone); file yang belum ada (misalnya di Colab) diunduh dari GitHub."""
    for f in FILE_KORPUS:
        tujuan = folder / f
        if not tujuan.exists():
            tujuan.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(f"{REPO_RAW}/{f}", tujuan)
    dokumen = {p.stem: p.read_text(encoding="utf-8") for p in sorted((folder / "dokumen").glob("*.md"))}
    baris = (folder / "pertanyaan_uji.jsonl").read_text(encoding="utf-8").splitlines()
    return dokumen, [json.loads(b) for b in baris]


dokumen, pertanyaan = muat_korpus()
print(f"{len(dokumen)} dokumen, {sum(len(t.split()) for t in dokumen.values()):,} kata | {len(pertanyaan)} pertanyaan uji")

Contoh isi satu dokumen (format Markdown dengan heading) dan contoh pertanyaan uji. Field `bukti` berisi kutipan persis dari dokumen yang dibutuhkan untuk menjawab. Kita memakainya untuk mengukur apakah retrieval menemukan chunk yang tepat. Pertanyaan berjenis `tidak_ada` sengaja tidak bisa dijawab dari dokumen.

In [ ]:
print(dokumen["02_garansi"][:700], "...\n")
pd.DataFrame(pertanyaan)[["id", "jenis", "pertanyaan", "bukti"]].head(8)

In [ ]:
pd.DataFrame(pertanyaan)["jenis"].value_counts()

## 1. Chunking: Memotong Dokumen

Mengapa dokumen perlu dipotong?
- Embedding satu vektor untuk dokumen panjang "merata-ratakan" banyak topik, sehingga maknanya kabur.
- Model embedding punya batas panjang input (e5: 512 token; sisanya terpotong).
- Konteks yang dikirim ke LLM sebaiknya ringkas dan relevan: lebih murah, dan LLM tidak "tersesat" dalam teks panjang.

Tiga strategi yang kita bandingkan:

| Strategi | Cara | Kelebihan / kekurangan |
|---|---|---|
| **Tetap** | tiap 60 kata, overlap 15 kata | Sederhana, tetapi bisa memotong di tengah kalimat atau mencampur dua topik |
| **Kalimat** | kumpulkan kalimat utuh sampai ±60 kata | Kalimat tidak terpotong |
| **Bagian** | satu chunk per heading `##`, diberi awalan `Judul > Subjudul` | Mengikuti struktur dokumen; awalan judul memberi konteks tambahan |

*Overlap* mengurangi risiko informasi penting terbelah di batas chunk.

In [ ]:
def chunk_tetap(dok_id, teks, n_kata=60, overlap=15):
    """Potong teks per n_kata kata; potongan berurutan berbagi `overlap` kata."""
    kata = teks.split()
    langkah = n_kata - overlap
    return [{"dok": dok_id, "teks": " ".join(kata[i:i + n_kata])}
            for i in range(0, max(len(kata) - overlap, 1), langkah)]


def chunk_kalimat(dok_id, teks, maks_kata=60):
    """Kumpulkan kalimat utuh sampai maks_kata kata per chunk."""
    baris = [b.strip("#- ").strip() for b in teks.splitlines() if b.strip()]
    kalimat = [k for b in baris for k in re.split(r"(?<=[.!?])\s+", b) if k]
    kelompok, sekarang = [], []
    for k in kalimat:
        if sekarang and len(" ".join(sekarang + [k]).split()) > maks_kata:
            kelompok.append(sekarang)
            sekarang = []
        sekarang.append(k)
    if sekarang:
        kelompok.append(sekarang)
    return [{"dok": dok_id, "teks": " ".join(g)} for g in kelompok]


def chunk_bagian(dok_id, teks, maks_kata=120):
    """Satu chunk per bagian (heading ##), diberi awalan "Judul > Subjudul" sebagai konteks."""
    judul = teks.splitlines()[0].lstrip("# ").strip()
    hasil = []
    for bagian in re.split(r"\n(?=## )", teks):
        baris = bagian.strip().splitlines()
        subjudul = baris[0].lstrip("# ").strip()
        isi = "\n".join(baris[1:]).strip()
        if not isi:
            continue
        konteks = judul if subjudul == judul else f"{judul} > {subjudul}"
        # Bagian yang terlalu panjang dipecah lagi, tetapi tetap membawa konteks judulnya
        potongan = [c["teks"] for c in chunk_tetap(dok_id, isi, maks_kata, 20)] if len(isi.split()) > maks_kata else [isi]
        hasil += [{"dok": dok_id, "teks": f"{konteks}\n{p}"} for p in potongan]
    return hasil


STRATEGI = {"tetap": chunk_tetap, "kalimat": chunk_kalimat, "bagian": chunk_bagian}
chunks_per_strategi = {nama: [c for d, t in dokumen.items() for c in f(d, t)] for nama, f in STRATEGI.items()}

pd.DataFrame([{"strategi": nama, "jumlah chunk": len(ch),
               "rata-rata kata": round(np.mean([len(c["teks"].split()) for c in ch]), 1),
               "maks kata": max(len(c["teks"].split()) for c in ch)}
              for nama, ch in chunks_per_strategi.items()])

Bandingkan chunk yang memuat informasi yang sama dari dua strategi. Perhatikan potongan **tetap** yang dimulai/diakhiri di tengah kalimat dan bisa mencampur bagian berbeda, sedangkan chunk **bagian** utuh dan membawa judulnya.

In [ ]:
for nama in ["tetap", "bagian"]:
    contoh = next(c for c in chunks_per_strategi[nama] if "Baterai hanya digaransi" in c["teks"])
    print(f"=== {nama} ===\n{contoh['teks']}\n")

## 2. Dense Retrieval dengan Embedding

Sama seperti modul 07, chunk dan pertanyaan diubah menjadi vektor, lalu dicari yang paling mirip (cosine similarity). Model **e5** dilatih dengan awalan khusus: `"query: "` untuk pertanyaan dan `"passage: "` untuk dokumen. Tanpa awalan ini kualitasnya turun. Selalu baca *model card* sebelum memakai model embedding.

Korpus kita kecil (puluhan chunk), jadi pencarian cukup dengan perkalian matriks NumPy. Untuk jutaan chunk, gunakan index ANN/vector database dari modul 07.

In [ ]:
EMBED_MODEL = "intfloat/multilingual-e5-small"
embedder = SentenceTransformer(EMBED_MODEL, device=device)
print(f"{EMBED_MODEL}: vektor {embedder.get_sentence_embedding_dimension()} dimensi, maks {embedder.max_seq_length} token")

def rapikan(teks):
    return " ".join(teks.split())


def token_bm25(teks):
    return re.findall(r"\w+", teks.lower())


class DenseRetriever:
    """Cari berdasarkan kemiripan makna (embedding e5). e5 butuh awalan 'query: ' dan 'passage: '."""

    def __init__(self, chunks):
        self.emb = embedder.encode(["passage: " + c["teks"] for c in chunks], normalize_embeddings=True)

    def skor(self, q):
        return self.emb @ embedder.encode("query: " + q, normalize_embeddings=True)

    def cari(self, q, k):
        return np.argsort(-self.skor(q))[:k].tolist()


chunks = chunks_per_strategi["bagian"]
dense = DenseRetriever(chunks)

q = "Apakah baterai HP saya masih ditanggung garansi setelah 5 bulan?"
for peringkat, i in enumerate(dense.cari(q, 3), 1):
    print(f"[{peringkat}] skor={dense.skor(q)[i]:.3f} | {chunks[i]['dok']} | {rapikan(chunks[i]['teks'])[:110]}…")

## 3. Mengukur Retrieval: Recall@k dan MRR

Untuk setiap pertanyaan yang bisa dijawab, kita tahu kutipan **bukti** yang harus ditemukan:
- **Recall@k**: proporsi bukti yang muncul di `k` chunk teratas. Ini metrik terpenting untuk RAG, karena LLM hanya melihat `k` chunk itu.
- **MRR@k** (*Mean Reciprocal Rank*): rata-rata `1 / peringkat` chunk pertama yang memuat bukti (0 jika tidak ada di `k` teratas). Makin tinggi, makin atas posisi chunk yang benar.

Pertanyaan berjenis `tidak_ada` tidak dihitung di sini (tidak ada bukti yang bisa ditemukan); perilakunya diuji saat generasi (notebook 02–03).

In [ ]:
DIJAWAB = [q for q in pertanyaan if q["bukti"]]
K_EVAL = (1, 3, 5)


def evaluasi(retriever, chunks, ks=K_EVAL):
    """Recall@k dan reciprocal rank untuk setiap pertanyaan yang bisa dijawab."""
    baris = []
    for q in DIJAWAB:
        urutan = retriever.cari(q["pertanyaan"], max(ks))
        memuat = [[rapikan(b) in rapikan(chunks[i]["teks"]) for b in q["bukti"]] for i in urutan]
        hasil = {"id": q["id"], "jenis": q["jenis"]}
        for k in ks:
            hasil[f"recall@{k}"] = np.mean([any(m[j] for m in memuat[:k]) for j in range(len(q["bukti"]))])
        posisi = next((r for r, m in enumerate(memuat, 1) if any(m)), None)
        hasil[f"MRR@{max(ks)}"] = 1 / posisi if posisi else 0.0
        baris.append(hasil)
    return pd.DataFrame(baris)


def ringkas(df):
    return df.drop(columns=["id", "jenis"]).mean().round(3)


hasil_chunking = pd.DataFrame({nama: ringkas(evaluasi(DenseRetriever(ch), ch))
                               for nama, ch in chunks_per_strategi.items()}).T
hasil_chunking

Dengan korpus sekecil ini selisih antarstrategi bisa kecil, tetapi polanya umum: chunk yang mengikuti struktur dokumen dan membawa judul biasanya lebih mudah ditemukan. Karena itu tidak ada ukuran chunk "terbaik" secara universal; **ukur pada data dan pertanyaanmu sendiri**. Selanjutnya kita pakai strategi `bagian`.

## 4. BM25 dan Hybrid Search

**BM25** adalah algoritma pencarian kata kunci klasik (dipakai mesin pencari sebelum era embedding): skor naik jika kata di pertanyaan muncul di dokumen, terutama kata yang jarang di seluruh korpus (mirip TF-IDF dari ML dasar).

| | Dense (embedding) | BM25 (kata kunci) |
|---|---|---|
| Kuat di | sinonim & parafrase ("ongkir" ≈ "ongkos kirim") | istilah persis, kode produk, angka, nama |
| Lemah di | istilah langka/spesifik yang jarang ada di data latih | kata berbeda dengan makna sama |

**Hybrid search** menggabungkan keduanya. Cara paling sederhana dan kuat adalah **Reciprocal Rank Fusion (RRF)**: setiap chunk mendapat skor `Σ 1 / (60 + peringkat)` dari tiap metode. RRF hanya memakai **peringkat**, sehingga tidak perlu menyamakan skala skor BM25 (tak terbatas) dan cosine (−1..1).

In [ ]:
class BM25Retriever:
    """Cari berdasarkan kecocokan kata (BM25)."""

    def __init__(self, chunks):
        self.bm25 = BM25Okapi([token_bm25(c["teks"]) for c in chunks])

    def skor(self, q):
        return self.bm25.get_scores(token_bm25(q))

    def cari(self, q, k):
        return np.argsort(-self.skor(q))[:k].tolist()


class HybridRetriever:
    """Gabungan dense + BM25 dengan Reciprocal Rank Fusion (RRF)."""

    def __init__(self, *retrievers, k_rrf=60, n_kandidat=50):
        self.retrievers, self.k_rrf, self.n_kandidat = retrievers, k_rrf, n_kandidat

    def cari(self, q, k):
        skor = defaultdict(float)
        for r in self.retrievers:
            for peringkat, i in enumerate(r.cari(q, self.n_kandidat), start=1):
                skor[i] += 1 / (self.k_rrf + peringkat)
        return sorted(skor, key=skor.get, reverse=True)[:k]


bm25 = BM25Retriever(chunks)
hybrid = HybridRetriever(dense, bm25)

per_metode = {"dense": evaluasi(dense, chunks), "bm25": evaluasi(bm25, chunks), "hybrid": evaluasi(hybrid, chunks)}
pd.DataFrame({nama: ringkas(df) for nama, df in per_metode.items()}).T

Rincian per jenis pertanyaan (recall@3). Pertanyaan `parafrase` memakai kata yang berbeda dari dokumen (misalnya "ongkir ke Makassar" vs "ongkos kirim ... luar Pulau Jawa"), sehingga di sinilah perbedaan dense dan BM25 biasanya paling terlihat.

In [ ]:
pd.DataFrame({nama: df.groupby("jenis")["recall@3"].mean().round(2) for nama, df in per_metode.items()})

## 5. Reranking dengan Cross-Encoder

Model embedding adalah **bi-encoder**: pertanyaan dan chunk di-encode **terpisah**, sehingga vektor chunk bisa dihitung sekali di awal (cepat), tetapi interaksi antara kata di pertanyaan dan kata di chunk hilang.

**Cross-encoder** membaca pasangan `(pertanyaan, chunk)` **sekaligus** dalam satu transformer, lalu mengeluarkan skor relevansi. Jauh lebih akurat, tetapi lambat: harus dijalankan untuk setiap pasangan saat ada pertanyaan. Solusinya adalah pipeline dua tahap:
1. Retriever cepat (hybrid) mengambil ±20 kandidat.
2. Cross-encoder mengurutkan ulang kandidat itu, lalu ambil `k` teratas.

In [ ]:
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"
reranker = CrossEncoder(RERANK_MODEL, max_length=512, device=device)


class RerankRetriever:
    def __init__(self, dasar, chunks, n_kandidat=20):
        self.dasar, self.chunks, self.n_kandidat = dasar, chunks, n_kandidat

    def cari(self, q, k):
        kandidat = self.dasar.cari(q, self.n_kandidat)
        skor = reranker.predict([(q, self.chunks[i]["teks"]) for i in kandidat])
        return [kandidat[j] for j in np.argsort(-skor)[:k]]


rerank = RerankRetriever(hybrid, chunks, n_kandidat=10 if QUICK_RUN else 20)
per_metode["hybrid + rerank"] = evaluasi(rerank, chunks)
tabel = pd.DataFrame({nama: ringkas(df) for nama, df in per_metode.items()}).T
tabel

In [ ]:
ax = tabel[["recall@1", "recall@3", "MRR@5"]].plot.bar(figsize=(9, 4), rot=0)
ax.set_ylim(0, 1.05)
ax.set_title("Kualitas retrieval per metode (chunking: bagian)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

Lihat pertanyaan yang masih gagal (recall@3 < 1) pada metode terbaik. Menganalisis kegagalan satu per satu sering memberi ide perbaikan (misalnya menambah sinonim, mengubah chunking, atau memperbaiki dokumen sumber).

In [ ]:
gagal = per_metode["hybrid + rerank"].query("`recall@3` < 1")
id_ke_q = {q["id"]: q for q in pertanyaan}
for _, row in gagal.iterrows():
    q = id_ke_q[row["id"]]
    print(f"[{q['id']}] {q['pertanyaan']}\n     bukti: {q['bukti']}\n     recall@3: {row['recall@3']:.2f}")
if gagal.empty:
    print("Semua bukti ditemukan di 3 chunk teratas.")

## 6. Retrieval Selalu Mengembalikan Sesuatu

Retriever **selalu** mengembalikan `k` chunk, bahkan untuk pertanyaan yang jawabannya tidak ada di korpus. Apakah skor similarity bisa dipakai untuk mendeteksinya?

In [ ]:
baris = []
for q in pertanyaan:
    skor = dense.skor(q["pertanyaan"])
    baris.append({"jenis": "tidak_ada" if q["jenis"] == "tidak_ada" else "bisa dijawab",
                  "skor dense teratas": skor.max(), "skor rerank teratas": reranker.predict(
                      [(q["pertanyaan"], chunks[i]["teks"]) for i in hybrid.cari(q["pertanyaan"], 5)]).max()})
pd.DataFrame(baris).groupby("jenis").agg(["mean", "min", "max"]).round(3)

Skor cosine e5 cenderung berkumpul di rentang sempit, sehingga ambang batas (threshold) sulit ditentukan dan rapuh. Skor reranker biasanya memisahkan lebih baik, tetapi tetap tidak sempurna. Dalam praktik, keputusan "informasinya tidak ada" diserahkan ke **LLM** dengan instruksi yang jelas (notebook 02), dan ambang skor dipakai sebagai pengaman tambahan.

## Ringkasan

- RAG = **retrieval** + **generation**. Kualitas retrieval adalah batas atas kualitas jawaban.
- **Chunking**: ikuti struktur dokumen bila ada, beri konteks (judul), dan pakai overlap untuk potongan tetap.
- **Dense** unggul untuk parafrase, **BM25** untuk istilah persis. **Hybrid (RRF)** menggabungkan keduanya tanpa perlu menyamakan skala skor.
- **Reranking** dengan cross-encoder meningkatkan urutan kandidat dengan biaya komputasi saat query.
- Ukur dengan **recall@k** dan **MRR** pada pertanyaan uji berlabel; analisis kegagalan satu per satu.

## Latihan

1. **Ukuran chunk.** Ulangi evaluasi dense untuk `chunk_tetap` dengan `n_kata` 20, 40, 60, 100, 200 (overlap 25%). Buat grafik recall@1 dan recall@3 terhadap ukuran chunk. Mengapa chunk yang terlalu kecil dan terlalu besar sama-sama buruk?
2. **Tanpa awalan e5.** Encode chunk dan pertanyaan tanpa awalan `"query: "`/`"passage: "`, lalu bandingkan recall-nya. Coba juga model embedding dari modul 07 (`paraphrase-multilingual-MiniLM-L12-v2`) dan perhatikan batas panjang inputnya (`max_seq_length`).
3. **Korpus sendiri.** Ambil 5 artikel Wikipedia bahasa Indonesia tentang satu topik (misalnya sejarah kota kelahiranmu), tulis 10 pertanyaan uji beserta kutipan bukti, lalu bandingkan dense, BM25, dan hybrid. Apakah kesimpulannya sama dengan korpus TEN?